In [1]:
!pip install transformers torch scikit-learn accelerate --quiet

In [2]:
pip install emoji

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 15.7 MB/s eta 0:00:00


In [3]:
import pandas as pd
import kagglehub
from datasets import load_dataset
import re
import emoji
import numpy as np
import torch
from transformers import EarlyStoppingCallback
from torch.utils.data import Dataset
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from sklearn.model_selection import train_test_split

# **Data 1**

In [6]:
path = kagglehub.dataset_download("monarasheedalroqi/arbcyd-arabic-cyberbullying-dataset")
print("Path to dataset files:", path)

Using Colab cache for faster access to the 'arbcyd-arabic-cyberbullying-dataset' dataset.
Path to dataset files: /kaggle/input/arbcyd-arabic-cyberbullying-dataset


In [7]:
data1=pd.read_csv(r"/kaggle/input/arbcyd-arabic-cyberbullying-dataset/ArbCyD Dataset.csv")

In [8]:
print("Head of data: ",data1.head())
print("\nData info ")
data1.info()
print("\n duplicate data",data1.duplicated().sum())
print("\n data is shape",data1.shape)
print(data1["Label"].value_counts())


Head of data:                                                Tweets  Domain         Label
0   لعلمكم فقط ان نزل لها نسخه مع انها ما فازت بذ...  Gaming  Non-bullying
1              واضح يوبي سوفت تبي تخربها ماتبي تعقل   Gaming  Non-bullying
2                                 اكثر يوتيبر محترم   Gaming  Non-bullying
3   نفس الحجة يستخدمونها على موضوع يقولون شوفوا إ...  Gaming  Non-bullying
4   واضح انه مدفوع لك، ولا عندك كرامه صدق ان لم ت...  Gaming      bullying

Data info 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Tweets  10000 non-null  object
 1   Domain  10000 non-null  object
 2   Label   10000 non-null  object
dtypes: object(3)
memory usage: 234.5+ KB

 duplicate data 11

 data is shape (10000, 3)
Label
Non-bullying    6204
bullying        3796
Name: count, dtype: int64


In [9]:
data1=data1.drop_duplicates()
print(data1.duplicated().sum())

0


In [10]:
print(data1["Label"].value_counts())

Label
Non-bullying    6197
bullying        3792
Name: count, dtype: int64


In [11]:
label_map={"Non-bullying":"Safe","bullying":"risky"}
data1["Label"]=data1["Label"].map(label_map)
print(data1["Label"].value_counts())

Label
Safe     6197
risky    3792
Name: count, dtype: int64


# **Data 2**

In [12]:
data2=pd.read_excel("/content/Arabic_offensive_comment_detection_annotation_4000_selected.xlsx")

In [13]:
print("Head of data: ",data2.head())
print("\nData info ")
data2.info()
print("\n duplicate data",data2.duplicated().sum())
print("\n data is shape",data2.shape)
print(data2["Majority_Label"].value_counts())


Head of data:     Id  Platform                                            Comment  \
0   1   Twitter  @User.IDX في فترة الصغر والمراهقة يكون من الصع...   
1   2  Facebook  "ردا على معظم الردود .. أحب اوضحلكم ان عمليات ...   
2   3   Twitter  @User.IDX يجب ان تذكروا ان لكل سنة ثيم للحفل و...   
3   4   YouTube  بتعمل حلقة صغيرة عشان عندي امتحان بكرة ومتضيعل...   
4   5   YouTube             على طاري السطحيه مدري ليه تذكرت فيحان    

  Majority_Label  Agreement  NumOfJudgementUsed  Total_Judgement  \
0  Non-Offensive      100.0                   3                3   
1  Non-Offensive      100.0                   3                4   
2  Non-Offensive      100.0                   3                5   
3  Non-Offensive      100.0                   3                3   
4  Non-Offensive      100.0                   3                3   

  Vulgar:V/HateSpeech:HS/None:-  
0                             -  
1                             -  
2                             -  
3                  

In [14]:
print(data2["Majority_Label"].value_counts())

Majority_Label
Non-Offensive    3325
Offensive         675
Name: count, dtype: int64


In [15]:
label_map={"Non-Offensive":"Safe","Offensive":"risky"}
data2["Majority_Label"]=data2["Majority_Label"].map(label_map)
print(data2["Majority_Label"].value_counts())

Majority_Label
Safe     3325
risky     675
Name: count, dtype: int64


# **Data 3**

In [16]:
ds = load_dataset("IbrahimAmin/egyptian-arabic-hate-speech")

README.md:   0%|          | 0.00/4.48k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  420kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  108kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/6535 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1634 [00:00<?, ? examples/s]

In [17]:
data3_train=ds["train"].to_pandas()
data3_test=ds["test"].to_pandas()

In [18]:
data3 = pd.concat([data3_train, data3_test], ignore_index=True)

In [19]:
print("Head of data: ",data3.head())
print("\nData info ")
data3.info()
print("\n duplicate data",data3.duplicated().sum())
print("\n data is shape",data3.shape)
print(data3_train["label"].value_counts())

Head of data:                                                  text                     label
0                                 انت في قمه النداله                 Offensive
1  يلعن دين المسلمين ويلعن دين الحركه الاسلاميه ك...  Religious Discrimination
2                   عيل زنجي عبد ابن كلب مشافش تربيه                    Racism
3    كسم ليبيا والعراق واللي منهم إرهابيين ولاد وسخة                    Racism
4                           بضان بضان يعني مفيش كلام                 Offensive

Data info 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8169 entries, 0 to 8168
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    8169 non-null   object
 1   label   8169 non-null   object
dtypes: object(2)
memory usage: 127.8+ KB

 duplicate data 0

 data is shape (8169, 2)
label
Neutral                     1708
Offensive                   1225
Racism                      1201
Sexism                      1201
Religious Discrimination

In [20]:
print(data3_train["label"].value_counts())

label
Neutral                     1708
Offensive                   1225
Racism                      1201
Sexism                      1201
Religious Discrimination    1200
Name: count, dtype: int64


In [21]:
label_map={"Neutral":"Safe","Offensive":"risky",
          "Racism":"risky","Sexism":"risky","Religious Discrimination":"risky" }
data3["label"]=data3["label"].map(label_map)
print(data3["label"].value_counts())

label
risky    6034
Safe     2135
Name: count, dtype: int64


# **Concation dataset togather**

In [22]:
data1=data1.rename(columns={"Tweets":"Feature",
                   "Label":"Target"})
data1=data1.drop(columns=["Domain"])

In [23]:
data2=data2.rename(columns={"Comment":"Feature",
                            "Majority_Label":"Target"})
data2=data2.drop(columns=["Id","Platform","Agreement","NumOfJudgementUsed"
,"Vulgar:V/HateSpeech:HS/None:-","Total_Judgement"])

In [24]:
data3=data3.rename(columns={"text":"Feature",
                            "label":"Target"})

In [25]:
All_data = pd.concat([data1, data2, data3], ignore_index=True)

In [26]:
All_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22158 entries, 0 to 22157
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   Feature  22158 non-null  object
 1   Target   22158 non-null  object
dtypes: object(2)
memory usage: 346.3+ KB


In [27]:
#All_data.to_csv("All_data.csv",index=False)

# **Preprocessing**

In [28]:
print("\nBefore duplicate data",All_data.duplicated().sum())
All_data=All_data.drop_duplicates()
print("\nAfter duplicate data",All_data.duplicated().sum())


Before duplicate data 1

After duplicate data 0


In [29]:
print("Count of missing value:",All_data.isnull().sum())

Count of missing value: Feature    0
Target     0
dtype: int64


In [30]:
def clean_arabic_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    text = re.sub(r"#", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    text = emoji.demojize(text, delimiters=(" ", " "))
    text = re.sub(r"[إأآا]", "ا", text)
    text = re.sub(r"ى", "ي", text)
    text = re.sub(r"[ؤئ]", "ء", text)
    text = re.sub(r"[\u064B-\u065F\u0670]", "", text)
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)
    text = re.sub(r"\s+", " ", text).strip()
    if len(text.split()) < 3:
        return ""
    return text

In [31]:
All_data['clean_text'] = All_data['Feature'].apply(clean_arabic_text)

In [32]:
print("\nBefore duplicate data",All_data.duplicated().sum())
print("Missing value \n",All_data.isnull().sum())


Before duplicate data 0
Missing value 
 Feature       0
Target        0
clean_text    0
dtype: int64


In [33]:
All_data[['Feature', 'clean_text', 'Target']].head(25)

,Feature,clean_text,Target
0,لعلمكم فقط ان نزل لها نسخه مع انها ما فازت بذ...,لعلمكم فقط ان نزل لها نسخه مع انها ما فازت بذي...,Safe
1,واضح يوبي سوفت تبي تخربها ماتبي تعقل,واضح يوبي سوفت تبي تخربها ماتبي تعقل,Safe
2,اكثر يوتيبر محترم,اكثر يوتيبر محترم,Safe
3,نفس الحجة يستخدمونها على موضوع يقولون شوفوا إ...,نفس الحجة يستخدمونها علي موضوع يقولون شوفوا اح...,Safe
4,واضح انه مدفوع لك، ولا عندك كرامه صدق ان لم ت...,واضح انه مدفوع لك، ولا عندك كرامه صدق ان لم تس...,risky
5,جيمر سناك، ماعندهم لا ولاء، او كرامه عادي جدا...,جيمر سناك، ماعندهم لا ولاء، او كرامه عادي جدا ...,risky
6,تذكروا انهم سحبوا منكم بطوله عشان كم شاذ وبدو...,تذكروا انهم سحبوا منكم بطوله عشان كم شاذ وبدون...,risky
7,حمار احمر شي,حمار احمر شي,risky
8,حمار صدق,,risky
9,حمار مسوي نفسه حصان,حمار مسوي نفسه حصان,risky


# Split data

In [34]:
All_data.dropna(subset=['Target'], inplace=True)
train_df, temp_df = train_test_split(All_data, test_size=0.2, stratify=All_data['Target'],random_state=42, shuffle=True)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['Target'], random_state=42, shuffle=True)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

Train: (17725, 3)
Val: (2216, 3)
Test: (2216, 3)


In [35]:
print("\nTrain:", train_df['Target'].value_counts())
print("\nVal:", val_df['Target'].value_counts())
print("\nTest:", test_df['Target'].value_counts())


Train: Target
Safe     9324
risky    8401
Name: count, dtype: int64

Val: Target
Safe     1166
risky    1050
Name: count, dtype: int64

Test: Target
Safe     1166
risky    1050
Name: count, dtype: int64


In [36]:
print("GPU متاح:", torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("الجهاز المستخدم:", device)

GPU متاح: True
الجهاز المستخدم: cuda


# **Model**

In [37]:
LABEL2ID = {"Safe": 0, "risky": 1}
ID2LABEL = {0: "Safe", 1: "risky"}

train_df['label_id'] = train_df['Target'].map(LABEL2ID)
val_df['label_id'] = val_df['Target'].map(LABEL2ID)
test_df['label_id'] = test_df['Target'].map(LABEL2ID)

In [38]:
model_name = "UBC-NLP/MARBERTv2"
tokenizer = AutoTokenizer.from_pretrained(model_name)


config.json:   0%|          | 0.00/757 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/439 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/1.10M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [39]:
class ArabicTextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.encodings = tokenizer(
            list(texts),
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt"
        )
        self.labels = list(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


In [40]:
train_dataset = ArabicTextDataset(train_df['clean_text'], train_df['label_id'], tokenizer)
val_dataset = ArabicTextDataset(val_df['clean_text'], val_df['label_id'], tokenizer)
test_dataset = ArabicTextDataset(test_df['clean_text'], test_df['label_id'], tokenizer)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

Train: 17725 | Val: 2216 | Test: 2216


In [41]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=2, id2label=ID2LABEL, label2id=LABEL2ID
)
model.to(device)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  654MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: UBC-NLP/MARBERTv2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were ne

model.safetensors: reconstructing file:   0%|          |  0.00B /  654MB            

model.safetensors: downloading bytes:           |  0.00B            

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(100000, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12

In [42]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary", pos_label=1
    )
    acc = accuracy_score(labels, preds)
    return {
        "accuracy": acc,
        "precision_risky": precision,
        "recall_risky": recall,
        "f1_risky": f1,
    }

In [43]:
training_args = TrainingArguments(
    output_dir="./marbert_risk_model",
    num_train_epochs=10,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_risky",
    greater_is_better=True,
    logging_steps=50,
    save_total_limit=2,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=3)]
)

In [44]:
print("\n🚀 بدء التدريب...")
history=trainer.train()


🚀 بدء التدريب...


Epoch,Training Loss,Validation Loss,Accuracy,Precision Risky,Recall Risky,F1 Risky
1,0.250845,0.207604,0.916516,0.880387,0.953333,0.915409
2,0.106109,0.389205,0.914711,0.869528,0.964762,0.914673
3,0.109754,0.397127,0.912455,0.906844,0.908571,0.907707
4,0.032470,0.587184,0.914260,0.917476,0.900000,0.908654


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [45]:
print("\n📊 تقييم على Test set...")
test_results = trainer.predict(test_dataset)
test_preds = np.argmax(test_results.predictions, axis=-1)
test_labels = test_df['label_id'].values

print("\n=== Classification Report ===")
print(classification_report(test_labels, test_preds, target_names=["Safe", "risky"]))

print("\n=== Confusion Matrix ===")
print(confusion_matrix(test_labels, test_preds))

acc = accuracy_score(test_labels, test_preds)
print(f"\nTest Accuracy: {acc:.4f}")


📊 تقييم على Test set...



=== Classification Report ===
              precision    recall  f1-score   support

        Safe       0.95      0.88      0.91      1166
       risky       0.87      0.95      0.91      1050

    accuracy                           0.91      2216
   macro avg       0.91      0.91      0.91      2216
weighted avg       0.92      0.91      0.91      2216


=== Confusion Matrix ===
[[1023  143]
 [  50 1000]]

Test Accuracy: 0.9129


In [46]:
trainer.save_model("/content/marbert_risk_model_final")
tokenizer.save_pretrained("/content/marbert_risk_model_final")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/marbert_risk_model_final/tokenizer_config.json',
 '/content/marbert_risk_model_final/tokenizer.json')